**Naive Bayes** is a fast, probabilistic classification algorithm based on Bayes' Theorem. It calculates the probability of a specific class (like Spam or Not Spam) given a set of observed features (like words in an email).

It assumes that all features are independent of each other.

# Terminologies

**Conditional Probability**: 

- It is the likelihood of an event happening given that another event has already occurred, written as $P(A\vert{}B)$ ("the probability of A given B")

- The Formula:
    - Written as: $P(A\vert{}B) = \frac{P(A \cap B)}{P(B)}$
    - $P(A \cap B)$: The probability of both event A and event B happening together.
    - $P(B)$: The probability of the given event B happening (must be greater than zero)


**Bayes' Theorem**

- Bayes' Theorem is a mathematical formula used to determine the conditional probability of an event based on prior knowledge and new evidence.

- The Formula
    $$P(A|B)=\frac{P(B|A)\cdot P(A)}{P(B)}$$
    
    - P(A|B) (Posterior): The probability that event A is true, given that B has happened.
    - P(B|A) (Likelihood): The probability of seeing evidence B if event A is true.
    - P(A) (Prior): Your initial guess or probability for event A before seeing new evidence.
    - P(B) (Marginal Likelihood): The total probability of the evidence B occurring across all possible outcomes.


- This formula is used in **Machine Learning** as:
    $$P(y\mid X)=\frac{P(X\mid y)\cdot P(y)}{P(X)}$$

    - $P(y \mid X)$ (The Posterior Probability):
        Example: What is the probability this email is Spam (y), given that it contains the word "Free" (X)?

    - $P(X \mid y)$ (The Likelihood):
        Example: If an email is already known to be Spam, how often does the word "Free" appear in it?

    - $P(y)$ (The Prior Probability):
        Example: Out of all emails ever received, what percentage are Spam? (e.g., If 20% of your inbox is spam, $P(\text{Spam}) = 0.2)$

    - $P(X)$ (The Predictor Prior / Marginal Likelihood):
        Example: Out of all emails ever received, what percentage contains thw word "Free"?

**Note**: The algorithm is called "Naive" because it assumes that every single feature is completely independent of the others given the class.




# Step-by-Step Workflow Example

Suppose we want to classify an incoming email containing the words ["Offer", "Free"] as Spam ($y=1$) or Ham/Not Spam ($y=0$).

1. **Compute Class Priors $P(y)$**
    If 30 out of 100 emails are Spam:
    $$P(\text{Spam}) = 0.30, \quad P(\text{Ham}) = 0.70$$

2. **Compute Feature Likelihoods $P(x_i \mid y)$**

    Calculate individual word frequencies within each class:
    - $P(\text{"Offer"} \mid \text{Spam}) = 0.60$, $\quad P(\text{"Free"} \mid \text{Spam}) = 0.80$
    - $P(\text{"Offer"} \mid \text{Ham}) = 0.10$, $\quad P(\text{"Free"} \mid \text{Ham}) = 0.05$

3. **Calculate Unnormalized Joint Probabilities**

    Formula: $P(X\mid y)\cdot P(y)$
    
    Spam Score: $P(\text{Spam}) \times P(\text{"Offer"} \mid \text{Spam}) \times P(\text{"Free"} \mid \text{Spam}) = 0.30 \times 0.60 \times 0.80 = \mathbf{0.144}$

    Ham Score: $P(\text{Ham}) \times P(\text{"Offer"} \mid \text{Ham}) \times P(\text{"Free"} \mid \text{Ham}) = 0.70 \times 0.10 \times 0.05 = \mathbf{0.0035}$

4. **Compute Marginal Likelihood $P(X)$:**
    
    $$P(X)=\text{Spam\ Top\ Part}+\text{Ham\ Top\ Part}$$

    $$P(X)=0.144+0.0035=\mathbf{0.1475}$$

5. **Compute Posterior Probability $P(y \mid X)$**

    - For Spam (y=1):
    $$P(\text{Spam}\mid \text{"Offer",\ "Free"})=\frac{0.144}{0.1475}=\mathbf{0.976}\text{\ (or\ }\mathbf{97.6\%)}$$
    
    - For Ham (y=0):
    $$P(\text{Ham}\mid \text{"Offer",\ "Free"})=\frac{0.0035}{0.1475}=\mathbf{0.024}\text{\ (or\ }\mathbf{2.4\%)}$$

    Since 97.6 > 2.4, the email is classified as Spam

**Note**: There are Three main Variants of Naive Bayes

- Gaussian Naive Bayes: Used for continuous features (e.g., age, height, temperature).
- Multinomial Naive Bayes: Used for discrete count data (e.g., word frequencies in document classification). Uses Laplace Smoothing to avoid multiplying by zero probabilities.
- Bernoulli Naive Bayes: Used for binary/boolean features (e.g., presence or absence of a feature: $1$ or $0$).

# Variants of Naive Bayes

What changes is how $P(x_i \mid y)$ is computed based on the data type of the input features:

|Variant|Data Type|Likelihood Formula $P(x_i \mid y)$|
|---|---|---|
|Categorical / Discrete (Used in Spam example)|Categories (e.g., "Free", "Paid")|Calculated directly from raw relative frequency counts: $$P(x_i \mid y) = \frac{\text{Count}(x_i \text{ in class } y)}{\text{Total items in class } y}$$​|
Multinomial|Word counts / term frequencies (0,1,2,5,…)|Uses word frequency counts with Laplace smoothing (α) to avoid zero probabilities: $$P(x_i \mid y)=\frac {N_{y​,i}+\alpha} {N_y ​+ \alpha \cdot d}​$$  $$P(\text{word} \mid \text{Class}) = \frac{\text{Count of that word in Class} + 1}{\text{Total count of all words in Class} + \text{Unique Vocabulary Size}}$$|
Bernoulli|Binary indicators (Feature present: 1, absent: 0)|Uses binary presence probabilities:$$P(x_i \mid y)=p_{i,y}^{x_i} ​​(1−p_{i,y}​)^{(1−x_i​)}$$|
Gaussian|Continuous numbers (e.g., Height, Salary, Age)|Fits a Bell Curve (Normal Distribution) using mean μy​ and variance $\sigma _{y}^2$​:$$P(x_i \mid y)=\frac {1} {\sqrt {2 \pi \sigma _{y}^2}}​exp(- \frac {(xi​−μy​)^2​} {2 \sigma _y^2})$$|

# Python Implementation

## Categorical / Discrete 

In [1]:
import numpy as np

class CategoricalNaiveBayes:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.classes = None
        self.log_priors = {}
        self.log_likelihoods = {}
        self.n_categories = {}

    def fit(self, X, y):
        X = np.asarray(X, dtype=int)
        y = np.asarray(y)
        
        n_samples, n_features = X.shape
        self.classes = np.unique(y)
        
        # Determine unique category counts for each feature
        for j in range(n_features):
            self.n_categories[j] = len(np.unique(X[:, j]))
        
        # Calculate Class Log Priors and Feature Log Likelihoods
        for c in self.classes:
            X_c = X[y == c]
            n_c = len(X_c)
            
            # Log Prior: log P(y = c)
            self.log_priors[c] = np.log(n_c / n_samples)
            self.log_likelihoods[c] = {}
            
            # Feature Likelihoods with Laplace Smoothing:
            # P(X_j = v | y = c) = (Count(X_j = v in c) + alpha) / (N_c + alpha * K_j)
            for j in range(n_features):
                K_j = self.n_categories[j]
                feature_counts = np.bincount(X_c[:, j], minlength=K_j)
                
                smoothed_probs = (feature_counts + self.alpha) / (n_c + self.alpha * K_j)
                self.log_likelihoods[c][j] = np.log(smoothed_probs)

        return self

    def predict_log_proba(self, X):
        X = np.asarray(X, dtype=int)
        n_samples, n_features = X.shape
        
        log_posteriors = np.zeros((n_samples, len(self.classes)))
        
        for idx, c in enumerate(self.classes):
            # Start with log prior
            log_prob = self.log_priors[c]
            
            # Sum log likelihoods across all features
            for j in range(n_features):
                feature_vals = X[:, j]
                log_prob += self.log_likelihoods[c][j][feature_vals]
                
            log_posteriors[:, idx] = log_prob
            
        return log_posteriors

    def predict(self, X):
        log_posteriors = self.predict_log_proba(X)
        class_indices = np.argmax(log_posteriors, axis=1)
        return self.classes[class_indices]


# TEST SCRIPT: HAND-CRAFTED CATEGORICAL DATASET

# Features: [Weather (0:Sunny, 1:Rainy), Temp (0:Hot, 1:Mild, 2:Cool), Wind (0:Weak, 1:Strong)]
# Target: Play Tennis (0: No, 1: Yes)

X_train = np.array([
    [0, 0, 0],  # Sunny, Hot, Weak   -> No
    [0, 0, 1],  # Sunny, Hot, Strong -> No
    [1, 1, 0],  # Rainy, Mild, Weak  -> Yes
    [1, 2, 0],  # Rainy, Cool, Weak  -> Yes
    [1, 2, 1],  # Rainy, Cool, Strong-> No
    [0, 1, 0],  # Sunny, Mild, Weak  -> Yes
])
y_train = np.array([0, 0, 1, 1, 0, 1])

# Test Queries
X_test = np.array([
    [0, 2, 0],  # Sunny, Cool, Weak   -> Expected: Likely Yes (1)
    [1, 0, 1],  # Rainy, Hot, Strong  -> Expected: Likely No (0)
])

# Instantiate and fit model
cnb = CategoricalNaiveBayes(alpha=1.0)
cnb.fit(X_train, y_train)

# Predictions and Log Posteriors
preds = cnb.predict(X_test)
log_probs = cnb.predict_log_proba(X_test)

print("=== CATEGORICAL NAIVE BAYES RESULTS ===")
for i, sample in enumerate(X_test):
    print(f"Query Sample {sample}:")
    print(f"  Log Posteriors [Class 0, Class 1]: {np.round(log_probs[i], 4)}")
    print(f"  Predicted Class: {preds[i]}\n")

=== CATEGORICAL NAIVE BAYES RESULTS ===
Query Sample [0 2 0]:
  Log Posteriors [Class 0, Class 1]: [-3.2189 -2.9312]
  Predicted Class: 1

Query Sample [1 0 1]:
  Log Posteriors [Class 0, Class 1]: [-2.8134 -4.6052]
  Predicted Class: 0



## MultiNomial

In [ ]:
import numpy as np


class MultinomialNaiveBayes:

  def __init__(self, alpha=1.0):
    self.alpha = alpha  # Laplace smoothing parameter
    self.classes = None
    self.log_priors = None
    self.log_likelihoods = None

  def fit(self, X, y):
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y)

    n_samples, n_features = X.shape
    self.classes = np.unique(y)
    n_classes = len(self.classes)

    self.log_priors = np.zeros(n_classes)
    self.log_likelihoods = np.zeros((n_classes, n_features))

    # Total vocabulary size (number of features d)
    d = n_features

    for idx, c in enumerate(self.classes):
      X_c = X[y == c]

      # Log Prior: log P(y = c) based on sample counts
      self.log_priors[idx] = np.log(len(X_c) / n_samples)

      # Word counts for class c: sum across rows for each feature j
      count_c_j = np.sum(X_c, axis=0)

      # Total words in class c across all features
      total_words_c = np.sum(count_c_j)

      # Multinomial Likelihood with Laplace Smoothing:
      # P(x_j | c) = (Count(w_j in c) + alpha) / (Total_Words_in_c + alpha * d)
      smoothed_probs = (count_c_j + self.alpha) / (total_words_c + self.alpha * d)
      self.log_likelihoods[idx, :] = np.log(smoothed_probs)

    return self

  def predict_log_proba(self, X):
    X = np.asarray(X, dtype=np.float64)

    # Matrix multiplication: X_test (M x d) @ log_likelihoods.T (d x |C|)
    # Resulting shape: (M, |C|)
    log_posteriors = X @ self.log_likelihoods.T + self.log_priors
    return log_posteriors

  def predict(self, X):
    log_posteriors = self.predict_log_proba(X)
    class_indices = np.argmax(log_posteriors, axis=1)
    return self.classes[class_indices]



# TEST SCRIPT: TEXT CLASSIFICATION (WORD COUNT VECTORS)

# Vocabulary: ["offer", "free", "money", "meeting", "project", "schedule"] (d = 6 features)
# Target: 0 = Ham (Legitimate), 1 = Spam

# Training Documents (Word Frequencies)
X_train = np.array([
    [3, 2, 1, 0, 0, 0],  # Spam email: heavy on offer/free/money
    [2, 3, 2, 0, 0, 0],  # Spam email
    [0, 0, 0, 2, 3, 1],  # Ham email: heavy on meeting/project/schedule
    [0, 1, 0, 1, 2, 2],  # Ham email
])
y_train = np.array([1, 1, 0, 0])

# Test Documents
X_test = np.array([
    [2, 1, 1, 0, 0, 0],  # New query: ["offer": 2, "free": 1, "money": 1] -> Expected: Spam (1)
    [0, 0, 0, 2, 1, 1],  # New query: ["meeting": 2, "project": 1, "schedule": 1] -> Expected: Ham (0)
])

# Instantiate & Fit Model
mnb = MultinomialNaiveBayes(alpha=1.0)
mnb.fit(X_train, y_train)

# Predictions
preds = mnb.predict(X_test)
log_probs = mnb.predict_log_proba(X_test)

print("=== MULTINOMIAL NAIVE BAYES RESULTS ===")
for i, doc in enumerate(X_test):
  print(f"Test Doc {i+1} Word Counts: {doc}")
  print(f"  Log Posteriors [Class 0 (Ham), Class 1 (Spam)]: {np.round(log_probs[i], 4)}")
  print(f"  Predicted Label: {'Spam' if preds[i] == 1 else 'Ham'}\n")

=== MULTINOMIAL NAIVE BAYES RESULTS ===
Test Doc 1 Word Counts: [2 1 1 0 0 0]
  Log Posteriors [Class 0 (Ham), Class 1 (Spam)]: [-11.5615  -5.7093]
  Predicted Label: Spam

Test Doc 2 Word Counts: [0 0 0 2 1 1]
  Log Posteriors [Class 0 (Ham), Class 1 (Spam)]: [ -6.304  -12.4709]
  Predicted Label: Ham



## Gaussian

In [ ]:
import numpy as np


class GaussianNaiveBayes:

  def __init__(self, var_smoothing=1e-9):
    self.var_smoothing = var_smoothing
    self.classes = None
    self.log_priors = None
    self.means = None
    self.vars = None

  def fit(self, X, y):
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y)

    n_samples, n_features = X.shape
    self.classes = np.unique(y)
    n_classes = len(self.classes)

    self.log_priors = np.zeros(n_classes)
    self.means = np.zeros((n_classes, n_features))
    self.vars = np.zeros((n_classes, n_features))

    for idx, c in enumerate(self.classes):
      X_c = X[y == c]

      # Log Prior: log P(y = c)
      self.log_priors[idx] = np.log(len(X_c) / n_samples)

      # Mean and Variance per feature for class c
      self.means[idx, :] = np.mean(X_c, axis=0)
      # Add var_smoothing epsilon to prevent division by zero
      self.vars[idx, :] = np.var(X_c, axis=0) + self.var_smoothing

    return self

  def predict_log_proba(self, X):
    X = np.asarray(X, dtype=np.float64)
    n_samples, _ = X.shape
    n_classes = len(self.classes)

    log_posteriors = np.zeros((n_samples, n_classes))

    for idx in range(n_classes):
      mean = self.means[idx]
      var = self.vars[idx]

      # Gaussian Log-PDF for all features simultaneously:
      # log P(x_j | c) = -0.5 * log(2 * pi * var_j) - ((x_j - mean_j)^2) / (2 * var_j)
      log_pdf = -0.5 * np.log(2 * np.pi * var) - ((X - mean) ** 2) / (2 * var)

      # Sum log likelihoods across features + add log prior
      log_posteriors[:, idx] = self.log_priors[idx] + np.sum(log_pdf, axis=1)

    return log_posteriors

  def predict(self, X):
    log_posteriors = self.predict_log_proba(X)
    class_indices = np.argmax(log_posteriors, axis=1)
    return self.classes[class_indices]


# TEST SCRIPT: CONTINUOUS PHYSICAL MEASUREMENTS

# Features: [Height (inches), Weight (lbs), Foot Size (inches)]
# Target: 0 = Female, 1 = Male

X_train = np.array([
    [66.0, 140.0, 9.0],  # Female
    [64.0, 120.0, 8.0],  # Female
    [62.0, 110.0, 7.0],  # Female
    [65.0, 130.0, 8.5],  # Female
    [72.0, 190.0, 11.0],  # Male
    [70.0, 175.0, 10.5],  # Male
    [68.0, 165.0, 10.0],  # Male
    [74.0, 210.0, 12.0],  # Male
])
y_train = np.array([0, 0, 0, 0, 1, 1, 1, 1])

# Query Test Points
X_test = np.array([
    [63.0, 115.0, 7.5],  # Short, lighter weight -> Expected: Female (0)
    [71.0, 180.0, 11.0],  # Taller, heavier weight -> Expected: Male (1)
])

# Instantiate and fit
gnb = GaussianNaiveBayes()
gnb.fit(X_train, y_train)

# Predictions & Scores
preds = gnb.predict(X_test)
log_probs = gnb.predict_log_proba(X_test)

print("=== GAUSSIAN NAIVE BAYES RESULTS ===")
for i, sample in enumerate(X_test):
  print(f"Query Sample {sample}:")
  print(
      "  Log Posteriors [Female (0), Male (1)]:"
      f" {np.round(log_probs[i], 4)}"
  )
  print(f"  Predicted Class: {'Male' if preds[i] == 1 else 'Female'}\n")

=== GAUSSIAN NAIVE BAYES RESULTS ===
Query Sample [ 63.  115.    7.5]:
  Log Posteriors [Female (0), Male (1)]: [ -7.068  -32.1196]
  Predicted Class: Female

Query Sample [ 71. 180.  11.]:
  Log Posteriors [Female (0), Male (1)]: [-36.0252  -6.8413]
  Predicted Class: Male



## Bernoulli

In [ ]:
import numpy as np


class BernoulliNaiveBayes:

  def __init__(self, alpha=1.0):
    self.alpha = alpha  # Laplace smoothing parameter
    self.classes = None
    self.log_priors = None
    self.feature_probs = None  # P(x_j = 1 | y = c)

  def fit(self, X, y):
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y)

    n_samples, n_features = X.shape
    self.classes = np.unique(y)
    n_classes = len(self.classes)

    self.log_priors = np.zeros(n_classes)
    self.feature_probs = np.zeros((n_classes, n_features))

    for idx, c in enumerate(self.classes):
      X_c = X[y == c]
      n_c = len(X_c)

      # Log Prior: log P(y = c)
      self.log_priors[idx] = np.log(n_c / n_samples)

      # P(x_j = 1 | c) with Laplace smoothing:
      # (Count(x_j = 1 in class c) + alpha) / (N_c + 2 * alpha)
      count_ones = np.sum(X_c, axis=0)
      self.feature_probs[idx, :] = (count_ones + self.alpha) / (
          n_c + 2 * self.alpha
      )

    return self

  def predict_log_proba(self, X):
    X = np.asarray(X, dtype=np.float64)
    n_samples, _ = X.shape
    n_classes = len(self.classes)

    log_posteriors = np.zeros((n_samples, n_classes))

    # Precompute log-probabilities for feature presence (1) and absence (0)
    log_p = np.log(self.feature_probs)  # log P(x_j = 1 | c)
    log_1_minus_p = np.log(1.0 - self.feature_probs)  # log P(x_j = 0 | c)

    for idx in range(n_classes):
      # Vectorized evaluation for class c across all samples:
      # log P(X | c) = X @ log(p_c) + (1 - X) @ log(1 - p_c)
      log_likelihood = X @ log_p[idx] + (1.0 - X) @ log_1_minus_p[idx]
      log_posteriors[:, idx] = self.log_priors[idx] + log_likelihood

    return log_posteriors

  def predict(self, X):
    log_posteriors = self.predict_log_proba(X)
    class_indices = np.argmax(log_posteriors, axis=1)
    return self.classes[class_indices]


# TEST SCRIPT: BINARY FEATURE MATRIX (SPAM DETECTION)

# Features: Presence (1) or Absence (0) of keywords:
# ["free", "money", "urgent", "meeting", "agenda"] (5 binary features)
# Target: 0 = Ham, 1 = Spam

X_train = np.array([
    [1, 1, 1, 0, 0],  # Spam: "free", "money", "urgent"
    [1, 1, 0, 0, 0],  # Spam: "free", "money"
    [1, 0, 1, 0, 0],  # Spam: "free", "urgent"
    [0, 0, 0, 1, 1],  # Ham:  "meeting", "agenda"
    [0, 0, 0, 1, 0],  # Ham:  "meeting"
    [0, 0, 1, 1, 1],  # Ham:  "urgent", "meeting", "agenda"
])
y_train = np.array([1, 1, 1, 0, 0, 0])

# Test Documents
X_test = np.array([
    [1, 1, 0, 0, 0],  # Contains ["free", "money"] -> Expected: Spam (1)
    [0, 0, 0, 1, 1],  # Contains ["meeting", "agenda"] -> Expected: Ham (0)
])

# Instantiate & Fit
bnb = BernoulliNaiveBayes(alpha=1.0)
bnb.fit(X_train, y_train)

# Predictions & Scores
preds = bnb.predict(X_test)
log_probs = bnb.predict_log_proba(X_test)

print("=== BERNOULLI NAIVE BAYES RESULTS ===")
for i, sample in enumerate(X_test):
  print(f"Sample {i+1} Feature Vector: {sample}")
  print(f"  Log Posteriors [Ham (0), Spam (1)]: {np.round(log_probs[i], 4)}")
  print(f"  Predicted Class: {'Spam' if preds[i] == 1 else 'Ham'}\n")

=== BERNOULLI NAIVE BAYES RESULTS ===
Sample 1 Feature Vector: [1 1 0 0 0]
  Log Posteriors [Ham (0), Spam (1)]: [-6.9486 -2.7897]
  Predicted Class: Spam

Sample 2 Feature Vector: [0 0 0 1 1]
  Log Posteriors [Ham (0), Spam (1)]: [-2.3842 -7.354 ]
  Predicted Class: Ham

